# SPARCS 2023 sepsis benchmarking

**Question:** which NY hospitals have more in-hospital sepsis deaths than expected, given how sick their patients are, and do those conclusions hold up under reasonable changes to the method?

**Plan:**
0. Setup
1. Load, explore, clean, build the cohort
2. EDA (including why raw death rates are misleading)
3. Risk models: each patient's chance of dying in hospital at an average NY hospital
4. Calibration
5. Hospital observed vs expected (O/E), with uncertainty and overdispersion
6. Robustness: outcome and feature choices, coding intensity, length of stay
7. Fairness check

Code lives in `src/sparcs/` (one module per step); `uv run sparcs` runs everything without this notebook. Model experiments are in `notebooks/modeling_lab.ipynb`. Column names follow the Socrata API; see `DATA_DICTIONARY.md`.

<!-- apr_drg_code: The All Patients Refined Diagnosis Related Groups (APR-DRG) Classification Code.
apr_drg_description: The APR-DRG Classification Code Description. -->

## 0. Setup
Imports, dataset settings, and folders for cached data and figures.

In [1]:
import os
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir("..")  # data/, figs/ and results/ paths are relative to the repo root

# Reload src/sparcs modules automatically when they change (no kernel restart needed)
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd

# Settings (dataset, paths) and one module per step
from sparcs import benchmark, checks, eda, models
from sparcs.config import APR_DRG, FIGS, RESULTS
from sparcs.data import build_cohort, clean, load_raw

FIGS.mkdir(exist_ok=True)
RESULTS.mkdir(exist_ok=True)

## 1. Load, explore, clean, build the cohort

### 1a. Load the raw data
Download once from the SPARCS API, then read the local Parquet copy.

In [2]:
# Downloads on the first run, then reads the cached copy in data/
raw = load_raw()
print(raw.shape)
print(sorted(raw.columns))

(131139, 33)
['age_group', 'apr_drg_code', 'apr_drg_description', 'apr_mdc_code', 'apr_mdc_description', 'apr_medical_surgical', 'apr_risk_of_mortality', 'apr_severity_of_illness', 'apr_severity_of_illness_code', 'birth_weight', 'ccsr_diagnosis_code', 'ccsr_diagnosis_description', 'ccsr_procedure_code', 'ccsr_procedure_description', 'discharge_year', 'emergency_department_indicator', 'ethnicity', 'facility_name', 'gender', 'hospital_county', 'hospital_service_area', 'length_of_stay', 'operating_certificate_number', 'patient_disposition', 'payment_typology_1', 'payment_typology_2', 'payment_typology_3', 'permanent_facility_id', 'race', 'total_charges', 'total_costs', 'type_of_admission', 'zip_code_3_digits']


### 1b. Explore the raw columns
Check each column's type, missingness and values before deciding what `clean()` should fix. Findings are noted at the top of `src/sparcs/data.py`.

In [3]:
# Show every column when printing wide tables
pd.set_option('display.max_columns', None)

In [4]:
raw.columns

Index(['hospital_service_area', 'hospital_county',
       'operating_certificate_number', 'permanent_facility_id',
       'facility_name', 'age_group', 'zip_code_3_digits', 'gender', 'race',
       'ethnicity', 'length_of_stay', 'type_of_admission',
       'patient_disposition', 'discharge_year', 'ccsr_diagnosis_code',
       'ccsr_diagnosis_description', 'ccsr_procedure_code',
       'ccsr_procedure_description', 'apr_drg_code', 'apr_drg_description',
       'apr_mdc_code', 'apr_mdc_description', 'apr_severity_of_illness_code',
       'apr_severity_of_illness', 'apr_risk_of_mortality',
       'apr_medical_surgical', 'payment_typology_1', 'payment_typology_2',
       'payment_typology_3', 'birth_weight', 'emergency_department_indicator',
       'total_charges', 'total_costs'],
      dtype='str')

In [5]:
def column_stats(df: pd.DataFrame, col: str, top: int = 15) -> None:
    s = df[col]
    print(f"=== {col} | dtype: {s.dtype}")
    print(f"missing: {s.isna().sum():,} ({s.isna().mean():.1%})")
    print(f"unique values: {s.nunique(dropna=True):,}")

    # handle the numeric cols
    if pd.api.types.is_numeric_dtype(s):
        print(f"min = {s.min()}")
        print(f"max = {s.max()}")
        print("summary:", s.describe().round(2).to_dict())

        q1, q3 = s.quantile([0.25, 0.75])
        iqr = q3 - q1
        outliers = s[(s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)]
        print(f"IQR outliers: {len(outliers):,}")

    # handle the non-numeric cols
    else:
        counts = s.value_counts(dropna=False)
        frequencies = pd.DataFrame({
            "n": counts,
            "percent": (counts / len(s) * 100).round(2),
        })
        print("most frequent values:")
        print(frequencies.head(top))

        # Diagnose strings that may represent numbers.
        as_num = pd.to_numeric(
            s.astype("string").str.replace(",", "", regex=False),
            errors="coerce",
        )
        if as_num.notna().mean() > 0.5:
            invalid = s[as_num.isna() & s.notna()]
            print("mostly numeric strings")
            print("values that do not convert:", invalid.value_counts().to_dict())

    print()

In [6]:
column_stats(raw, 'apr_risk_of_mortality')

=== apr_risk_of_mortality | dtype: str
missing: 0 (0.0%)
unique values: 4
most frequent values:
                           n  percent
apr_risk_of_mortality                
Extreme                58303    44.46
Major                  40943    31.22
Moderate               19016    14.50
Minor                  12877     9.82



### 1c. Clean and build the cohort
`clean()` fixes types and adds the two outcomes; it drops no rows.
`build_cohort()` applies the exclusions in order and counts what's left after each one:
1. No facility id (redacted records can't be benchmarked)
2. Children (0 to 17)
3. Transfers out to another hospital (final outcome unknown)
4. Left against medical advice (care cut short, outcome unknown)
5. Hospitals whose sepsis admissions are over 90% elective. Sepsis is an emergency, so these hospitals are admitting for something else (palliative or rehab care). SPARCS and the NY facility registry list every hospital as just "Hospital", so this is found from the data.
6. Hospitals reporting 0 deaths among 100+ sepsis discharges (implausible at ~12% mortality; points to a disposition reporting problem)

Outcomes (both are reported; see step 6):
- `died`: died in hospital ("Expired")
- `died_or_hospice`: died, or discharged to hospice. Hospitals differ in how often dying patients go to hospice, which can make in-hospital mortality look better than it is.

In [7]:
# Fix types, add outcomes, then apply the cohort exclusions
df, cohort_steps = build_cohort(clean(raw))

# CONSORT-style table: rows left after each exclusion
print(cohort_steps)
print(f"Cohort: {len(df):,} discharges, {df.permanent_facility_id.nunique()} hospitals, "
      f"mortality {df.died.mean():.1%}, died or hospice {df.died_or_hospice.mean():.1%}")

                                                step       n
0                              all sepsis discharges  131139
1                                facility identified  131046
2                                             adults  128637
3                not transferred to another hospital  124939
4               did not leave against medical advice  121801
5  hospital treats acute sepsis (not mostly elect...  121728
6                  hospital reports deaths plausibly  120995
Cohort: 120,995 discharges, 180 hospitals, mortality 12.6%, died or hospice 16.5%


## 2. EDA
Describe the cohort before modelling it. The functions live in `src/sparcs/eda.py`; new checks get added here as later steps raise questions.

### 2a. Outcome balance
How rare is death? This decides whether imbalance needs handling.

In [8]:
# About 1 in 8 patients die. Imbalanced, but not rare: no resampling (SMOTE,
# oversampling), because it would distort the predicted probabilities and
# calibration is what the O/E ratios depend on.
eda.outcome_balance(df)

,n,rate
died,15199,0.126
died_or_hospice,19959,0.165


### 2b. Missing data
Which columns have gaps, and do any of them matter for the model?

In [9]:
# None of the model features have gaps. Procedure codes are most likely blank
# when no procedure was done, and payer 2/3 when there was only one payer, so
# those gaps probably mean "none" rather than "unknown" (check the data
# dictionary before using them). birth_weight is only recorded for newborns.
eda.missing_table(df)

,missing,percent
birth_weight,120995,100.0
payment_typology_3,102570,84.8
payment_typology_2,48271,39.9
ccsr_procedure_code,42480,35.1
ccsr_procedure_description,42480,35.1
zip_code_3_digits,1045,0.9


### 2c. Mortality by each candidate feature
A sanity check before modelling: does each feature separate high- and low-risk patients, and are any categories too small to estimate?

In [10]:
# Candidate risk-model inputs: patient factors, not hospital ones.
# Length of stay and cost are left out: they are only known after the stay.
# Caveat: APR severity and risk of mortality are assigned after discharge from
# all coded diagnoses, so they can include complications that started in hospital.
CANDIDATE_FEATURES = ["age_group", "gender", "type_of_admission",
                      "emergency_department_indicator", "apr_severity_of_illness_code",
                      "apr_risk_of_mortality", "payment_typology_1", "ccsr_diagnosis_description"]

for col in CANDIDATE_FEATURES:
    display(eda.mortality_by(df, col))

,n,mortality
age_group,,
18 to 29,3984,0.020
30 to 49,13359,0.050
50 to 69,37270,0.101
70 or Older,66382,0.161


,n,mortality
gender,,
F,61074,0.124
M,59919,0.127
U,2,0.000


,n,mortality
type_of_admission,,
Elective,863,0.213
Emergency,115944,0.122
Newborn,1,0.000
Not Available,104,0.077
Trauma,147,0.170
Urgent,3936,0.199


,n,mortality
emergency_department_indicator,,
N,5780,0.204
Y,115215,0.122


,n,mortality
apr_severity_of_illness_code,,
1,4011,0.002
2,23431,0.005
3,41978,0.046
4,51575,0.255


,n,mortality
apr_risk_of_mortality,,
Extreme,54623,0.253
Major,38085,0.035
Minor,11086,0.000
Moderate,17201,0.005


,n,mortality
payment_typology_1,,
Blue Cross/Blue Shield,6488,0.068
Department of Corrections,75,0.093
Federal/State/Local/VA,1210,0.141
"Managed Care, Unspecified",934,0.086
Medicaid,20247,0.080
Medicare,80869,0.148
Miscellaneous/Other,431,0.146
Private Health Insurance,9718,0.079
Self-Pay,1023,0.108


,n,mortality
ccsr_diagnosis_description,,
Bacterial infections,47,0.191
"Complication of other surgical or medical care, injury, initial encounter",493,0.026
Foodborne intoxications,2,0.000
Septicemia,120439,0.126
Tuberculosis,14,0.143


### 2d. How related are the features? (Cramér's V)
Correlated features don't hurt prediction or calibration, but they share credit, so their individual odds ratios are harder to interpret.

In [11]:
# Expect severity <-> risk of mortality (same APR grouper), admission type <-> ED
# (emergencies mostly come through the ED), and age <-> payer (Medicare at 65+)
eda.cramers_v_matrix(df, CANDIDATE_FEATURES)

,age_group,gender,type_of_admission,emergency_department_indicator,apr_severity_of_illness_code,apr_risk_of_mortality,payment_typology_1,ccsr_diagnosis_description
age_group,1.00,0.04,0.02,0.03,0.16,0.21,0.37,0.02
gender,0.04,1.00,0.01,0.02,0.03,0.03,0.06,0.01
type_of_admission,0.02,0.01,1.00,0.60,0.02,0.02,0.03,0.02
emergency_department_indicator,0.03,0.02,0.60,1.00,0.03,0.03,0.04,0.03
apr_severity_of_illness_code,0.16,0.03,0.02,0.03,1.00,0.59,0.14,0.02
apr_risk_of_mortality,0.21,0.03,0.02,0.03,0.59,1.00,0.17,0.02
payment_typology_1,0.37,0.06,0.03,0.04,0.14,0.17,1.00,0.02
ccsr_diagnosis_description,0.02,0.01,0.02,0.03,0.02,0.02,0.02,1.00


### 2e. Severity, length of stay, hospital size

In [12]:
# Mortality by risk of mortality, length of stay, discharges per hospital
eda.plot_eda(df, FIGS / "eda.png")

# Mortality, cost and length of stay by severity of illness
display(eda.severity_table(df))

# Many hospitals are small, so their O/E will be noisy (handled by the funnel plot)
eda.hospital_sizes(df)

,n,mortality,median_cost,median_los
apr_severity_of_illness_code,,,,
1,4011,0.002,7976.780,3.0
2,23431,0.005,10492.060,4.0
3,41978,0.046,16830.615,5.0
4,51575,0.255,29750.630,8.0


hospitals          180
under 50 cases      29
under 100 cases     34
under 300 cases     61
median cases       510
dtype: int64

### 2f. Why not just compare raw death rates?
Hospitals see different patients. If a hospital's raw mortality mostly follows its share of very sick patients, ranking hospitals on raw rates measures case mix, not care. That's what risk adjustment fixes.

In [13]:
# Hospitals with 300+ cases (smaller ones have noisy rates), sorted by share of Extreme-risk patients
case_mix = eda.case_mix_table(df, min_n=300)
corr = case_mix["raw_mortality"].corr(case_mix["share_extreme"])
print(f"{len(case_mix)} hospitals with 300+ cases; "
      f"correlation of raw mortality with share Extreme = {corr:.2f}")

# Hospitals with the least and most severe patient mix
pd.concat([case_mix.head(3), case_mix.tail(3)]).round(3)

119 hospitals with 300+ cases; correlation of raw mortality with share Extreme = 0.53


,,n,raw_mortality,share_extreme
permanent_facility_id,facility_name,,,
000671,Geneva General Hospital,380,0.113,0.313
000628,UPSTATE University Hospital at Community General,753,0.046,0.317
000818,Saratoga Hospital,975,0.101,0.326
001165,Jacobi Medical Center,799,0.199,0.584
003975,New York-Presbyterian Hospital - Allen Hospital,720,0.171,0.644
001738,Richmond University Medical Center,493,0.239,0.649


### 2g. Is sepsis coded the same way everywhere?
Sepsis here is defined by billing codes. A hospital that codes sepsis liberally adds milder patients and looks better on raw mortality, with no change in care. An unusually high share of mild (severity 1-2) sepsis is a hint of that, not proof: some hospitals really do see milder sepsis.

In [14]:
# Share of each hospital's sepsis patients coded as mild, for hospitals with 100+ cases
mild = eda.mild_sepsis_share(df, min_n=100)
print(f"{len(mild)} hospitals; share mild ranges {mild.share_mild.min():.0%} to "
      f"{mild.share_mild.max():.0%}; correlation with raw mortality = "
      f"{mild.share_mild.corr(mild.raw_mortality):.2f}")

# Highest and lowest mild share
pd.concat([mild.head(3), mild.tail(3)]).round(3)

146 hospitals; share mild ranges 4% to 38%; correlation with raw mortality = -0.34


,,n,share_mild,raw_mortality
permanent_facility_id,facility_name,,,
001318,Wyckoff Heights Medical Center,728,0.376,0.102
000528,Nassau University Medical Center,650,0.375,0.155
000704,St Anthony Community Hospital,109,0.367,0.165
001463,NYU Langone Hospitals,2041,0.122,0.049
000885,Long Island Community Hospital at NYU Langone Health,1422,0.095,0.088
001453,Memorial Hospital for Cancer and Allied Diseases,1065,0.040,0.182


## 3. Risk models
Predict each patient's chance of dying in hospital at an average NY hospital, from patient factors only. The model is only a tool for expected deaths; nothing is predicted at admission. Code: `src/sparcs/models.py`.

**Feature rule:** a variable is allowed if the hospital's care can't change it (illness the patient arrived with), not because of when it's recorded.
- Out: hospital (shows up later as O/E), race (used for the fairness check), length of stay and cost (shaped by care).
- Severity of illness and risk of mortality (ROM) are coded at discharge and can absorb complications, so `admission_only` is kept as the other extreme.

Predictions are out-of-fold, split by hospital: each hospital is scored by a model that never saw its patients.

In [15]:
# Feature sets, one per sensitivity question (reasoning in models.FEATURE_SETS)
for name, cols in models.FEATURE_SETS.items():
    print(f"{name:16s} {cols}")

# Logistic regression on every feature set, LightGBM on the full set only
preds = models.fit_all(df)
y = df["died"].to_numpy()

# Discrimination (AUROC, AUPRC), accuracy of the probabilities (Brier), calibration
metrics = models.metrics_table(y, preds)
metrics.to_csv(RESULTS / "model_metrics.csv", index=False)
metrics

full             ['age_group', 'gender', 'type_of_admission', 'emergency_department_indicator', 'apr_severity_of_illness_code', 'apr_risk_of_mortality']
no_rom           ['age_group', 'gender', 'type_of_admission', 'emergency_department_indicator', 'apr_severity_of_illness_code']
full_plus_payer  ['age_group', 'gender', 'type_of_admission', 'emergency_department_indicator', 'apr_severity_of_illness_code', 'apr_risk_of_mortality', 'payment_typology_1']
full_plus_dx     ['age_group', 'gender', 'type_of_admission', 'emergency_department_indicator', 'apr_severity_of_illness_code', 'apr_risk_of_mortality', 'ccsr_diagnosis_description']
admission_only   ['age_group', 'gender', 'type_of_admission', 'emergency_department_indicator']


,model,auroc,auprc,brier,cal_intercept,cal_slope
0,lr_full,0.8083,0.2956,0.0939,-0.0025,0.9896
1,lr_no_rom,0.7842,0.2714,0.0964,-0.0026,0.9906
2,lr_full_plus_payer,0.8083,0.2959,0.0939,-0.0035,0.9851
3,lr_full_plus_dx,0.8085,0.2960,0.0939,-0.0023,0.9889
4,lr_admission_only,0.6016,0.1629,0.1077,-0.0018,0.9679
5,lgbm_full,0.8078,0.2923,0.0939,-0.0023,0.9716


### 3b. What drives risk? Odds ratios
Unpenalized logistic regression on the main feature set, with 95% CIs. Each odds ratio compares a group with its reference (e.g. ages 70+ vs 18 to 29). Severity and ROM are strongly related (Cramér's V 0.59), so they share credit: read them together, not one at a time.

In [16]:
odds = models.odds_ratios(df, models.FEATURE_SETS["full"])
print(f"rows dropped (categories under 50 patients): {odds.attrs['rows_dropped']}")
odds

rows dropped (categories under 50 patients): 3


odds_ratio   ci_low   ci_high  \
feature                        level                                          
age_group                      30 to 49           2.1073   1.6465    2.6970   
                               50 to 69           2.8883   2.2828    3.6544   
                               70 or Older        4.1416   3.2787    5.2316   
gender                         M                  0.9598   0.9250    0.9960   
type_of_admission              Elective           1.4612   1.1924    1.7907   
                               Not Available      0.4700   0.2162    1.0220   
                               Trauma             0.8480   0.5415    1.3279   
                               Urgent             1.1697   1.0410    1.3144   
emergency_department_indicator Y                  0.6129   0.5534    0.6789   
apr_severity_of_illness_code   2                  0.2897   0.1384    0.6065   
                               3                  0.6370   0.3042    1.3337   
                               4                  1.7885   0.8531    3.7492   
apr_risk_of_mortality          Extreme          227.5903  78.0758  663.4240   
                               Major             50.2513  17.2513  146.3765   
                               Moderate          12.7183   4.3897   36.8490   

                                              p_value  
feature                        level                   
age_group                      30 to 49        0.0000  
                               50 to 69        0.0000  
                               70 or Older     0.0000  
gender                         M               0.0297  
type_of_admission              Elective        0.0003  
                               Not Available   0.0568  
                               Trauma          0.4711  
                               Urgent          0.0084  
emergency_department_indicator Y               0.0000  
apr_severity_of_illness_code   2               0.0010  
                               3               0.2316  
                               4               0.1237  
apr_risk_of_mortality          Extreme         0.0000  
                               Major           0.0000  
                               Moderate        0.0000

## 4. Calibration
Do predicted risks match what actually happened? This matters more than AUROC here, because expected deaths are sums of predicted probabilities.

Two numbers summarize the plot:
- **Calibration intercept** ~0: predictions aren't too high or too low overall.
- **Calibration slope** ~1: predictions aren't too extreme (<1) or too timid (>1).

In [17]:
# Deciles of predicted risk: mean predicted vs observed mortality
models.plot_calibration(y, preds, FIGS / "calibration.png")
metrics.set_index("model")[["cal_intercept", "cal_slope"]]

,cal_intercept,cal_slope
model,,
lr_full,-0.0025,0.9896
lr_no_rom,-0.0026,0.9906
lr_full_plus_payer,-0.0035,0.9851
lr_full_plus_dx,-0.0023,0.9889
lr_admission_only,-0.0018,0.9679
lgbm_full,-0.0023,0.9716


## 5. Hospital observed vs expected (O/E) and funnel plot
- Observed (O) = actual deaths; expected (E) = sum of patients' predicted risks. O/E above 1 means more deaths than the patient mix explains.
- Each O/E gets an exact 95% CI.
- Small hospitals vary by chance, so flags use limits that widen as E shrinks:
  - **Poisson limits:** chance only.
  - **Adjusted limits:** chance plus the usual variation between hospitals (overdispersion; Spiegelhalter 2005). A hospital outside these stands out even from that normal spread.
- Many hospitals means some get flagged by chance alone; the 99.8% limit is the one that counts.
- Code: `src/sparcs/benchmark.py`.

In [18]:
# O/E per hospital from the main model, with 95% CI and both flags
hosp = benchmark.hospital_oe(df, preds["lr_full"])
hosp.sort_values("oe", ascending=False).to_csv(RESULTS / "hospital_oe.csv", index=False)

# Do hospitals vary more than chance allows? phi well above 1 (tau2 > 0) says yes
od = hosp.attrs["overdispersion"]
print(f"phi = {od['phi']:.2f}, tau2 = {od['tau2']:.4f}, p = {od['p_value']:.1e}")

# Hospitals flagged by chance alone if none truly differed
print("expected false flags:",
      {k: round(v, 1) for k, v in benchmark.expected_false_flags(len(hosp)).items()})

benchmark.plot_funnel(hosp, FIGS / "funnel_plot.png",
                      f"Sepsis mortality O/E by hospital, NY 2023 (APR-DRG {APR_DRG})")

# Poisson flags (rows) vs overdispersion-adjusted flags (columns)
pd.crosstab(hosp["flag"], hosp["flag_adjusted"], margins=True)

phi = 4.87, tau2 = 0.0463, p = 2.2e-92
expected false flags: {'95%': 9.0, '99.8%': 0.4}


flag_adjusted,high (95%),high (99.8%),low (95%),within,All
flag,,,,,
high (95%),4,0,0,12,16
high (99.8%),12,5,0,11,28
low (95%),0,0,0,10,10
low (99.8%),0,0,8,10,18
within,0,0,0,108,108
All,16,5,8,151,180


In [19]:
# The strongest signals: outside the adjusted 99.8% limits
show = ["facility_name", "n", "observed", "expected", "oe", "oe_ci_low", "oe_ci_high", "flag"]
hosp[hosp["flag_adjusted"].str.contains("99.8")].sort_values("oe", ascending=False)[show].round(2)

,facility_name,n,observed,expected,oe,oe_ci_low,oe_ci_high,flag
7,"Brooks-TLC Hospital System, Inc.",169,32,12.92,2.48,1.69,3.50,high (99.8%)
116,St. Joseph's Medical Center,177,33,16.29,2.03,1.39,2.85,high (99.8%)
164,St Johns Episcopal Hospital So Shore,852,207,102.44,2.02,1.75,2.32,high (99.8%)
132,Brooklyn Hospital Center - Downtown Campus,565,116,59.29,1.96,1.62,2.35,high (99.8%)
131,Brookdale Hospital Medical Center,431,117,60.55,1.93,1.60,2.32,high (99.8%)


## 6. Robustness
Two method choices could drive the flags instead of real differences in care:
- **Outcome:** in-hospital death misses patients discharged to die elsewhere, so `died_or_hospice` is a co-primary outcome.
- **Features:** severity and risk of mortality are coded at discharge and can absorb complications, so `admission_only` is the other extreme.

A hospital flagged in the same direction under all four combinations is a **robust signal**. Code: `src/sparcs/checks.py`.

### 6a. Do hospitals keep the same order? (Spearman rank correlation of O/E)

In [20]:
# Predictions for every outcome x feature-extreme combination
variants = checks.variant_predictions(df)

# Rank agreement across feature sets and the hospice outcome; near 1 = same order
checks.rank_agreement(df, preds, variants[("died_or_hospice", "full")])

,full,no_rom,full_plus_payer,full_plus_dx,admission_only,died_or_hospice
full,1.000,0.993,1.000,1.000,0.798,0.893
no_rom,0.993,1.000,0.993,0.992,0.825,0.885
full_plus_payer,1.000,0.993,1.000,1.000,0.799,0.893
full_plus_dx,1.000,0.992,1.000,1.000,0.797,0.892
admission_only,0.798,0.825,0.799,0.797,1.000,0.711
died_or_hospice,0.893,0.885,0.893,0.892,0.711,1.000


### 6b. Robust flags
Adjusted (overdispersion) flags under each variant. "high" or "low" in the robust column only if all four agree.

In [21]:
for level in ["95%", "99.8%"]:
    robust = checks.robust_flags(df, variants, level)
    robust.to_csv(RESULTS / f"robust_flags_{level.rstrip('%')}.csv")
    print(f"robust at adjusted {level}:", robust["robust"].value_counts().to_dict())

# The robust signals at 95%, with O/E under each variant
robust = checks.robust_flags(df, variants, "95%")
robust[robust["robust"] != "not robust or within"]["oe"].round(2).assign(robust=robust["robust"])

robust at adjusted 95%: {'not robust or within': 171, 'high': 7, 'low': 2}
robust at adjusted 99.8%: {'not robust or within': 178, 'high': 2}


,,died / full,died / admission_only,died_or_hospice / full,died_or_hospice / admission_only,robust
permanent_facility_id,facility_name,,,,,
000267,Kenmore Mercy Hospital,1.64,1.87,1.36,1.54,high
000511,NYU Langone Hospital-Long Island,0.40,0.40,0.59,0.58,low
000739,A.O. Fox Memorial Hospital,0.31,0.20,0.42,0.28,low
000925,Good Samaritan Hospital Medical Center,1.54,1.57,1.49,1.52,high
001139,Westchester Medical Center,1.70,1.75,1.54,1.58,high
001286,Brookdale Hospital Medical Center,1.93,2.38,1.61,1.94,high
001288,Brooklyn Hospital Center - Downtown Campus,1.96,1.70,1.72,1.51,high
001294,NYC Health + Hospitals/South Brooklyn Health,1.61,1.70,1.38,1.44,high
001635,St Johns Episcopal Hospital So Shore,2.02,2.12,1.55,1.61,high


### 6c. Coding intensity
Expected deaths with discharge-coded severity, divided by expected deaths from admission factors alone. A high ratio means sicker patients or more thorough coding (the data can't tell which). If heavy coding drove the results, hospitals with high ratios would have low O/E.

In [22]:
coding = checks.coding_intensity(df, variants[("died", "full")], variants[("died", "admission_only")])
print(f"Spearman(coding ratio, O/E) = {coding['coding_ratio'].corr(coding['oe'], method='spearman'):.2f}")
coding.sort_values("coding_ratio", ascending=False).head().round(2)

Spearman(coding ratio, O/E) = 0.09


,,observed,e_full,e_adm,coding_ratio,oe
permanent_facility_id,facility_name,,,,,
000208,John R. Oishei Children's Hospital,1,0.67,0.18,3.76,1.50
001446,NYU Langone Orthopedic Hospital,4,7.72,5.39,1.43,0.52
003975,New York-Presbyterian Hospital - Allen Hospital,123,126.19,93.52,1.35,0.97
001738,Richmond University Medical Center,118,82.42,61.44,1.34,1.43
015464,"Lockport Memorial Hospital, a Campus of Mount St Mary`s",0,0.54,0.41,1.31,0.00


### 6d. Discharge practice: length of stay
Shorter stays leave less time for deaths to happen in hospital. If short-stay hospitals had low O/E, discharge practice rather than care could explain part of their result.

In [23]:
los = checks.los_vs_oe(df, hosp)
print(f"Spearman(median LOS, O/E) = {los['median_los'].corr(los['oe'], method='spearman'):.2f}")
los.sort_values("median_los").head().round(2)

Spearman(median LOS, O/E) = 0.20


,facility_name,n,oe,median_los
permanent_facility_id,,,,
001315,Kingsbrook Jewish Medical Center,3,7.65,1.0
000111,Westfield Memorial Hospital Inc,10,4.34,2.0
000873,Ira Davenport Memorial Hospital,22,3.38,2.0
000303,The University of Vermont Health Network - Eli...,42,2.90,3.0
000891,Stony Brook Eastern Long Island Hospital,93,0.78,3.0


## 7. Fairness check
Race is not a model input. Compare observed and expected deaths within each group: O/E near 1 (with its 95% CI covering 1) means the model isn't systematically over- or under-predicting risk for that group.

An overall gap can have two causes: the model misjudges the group, or the group is treated more often at hospitals with high O/E. `oe_within_hospital` separates them by giving each patient their own hospital's O/E.

In [24]:
fair = checks.calibration_by_race(df, preds["lr_full"])
fair.to_csv(RESULTS / "fairness_by_race.csv")
fair

,n,observed,expected,oe,oe_ci_low,oe_ci_high,oe_within_hospital
race,,,,,,,
Black/African American,17678,2423,2161.094,1.121,1.077,1.167,1.020
Multi-racial,1420,163,186.078,0.876,0.747,1.021,0.993
Other Race,28809,3367,3343.578,1.007,0.973,1.042,0.977
White,73088,9246,9536.581,0.970,0.950,0.989,1.003


### 7b. Same question as a regression
Odds of death by race vs White patients, beyond the model's predicted risk, with and without comparing patients inside the same hospital. If an odds ratio shrinks to ~1 once hospital is added, the group's extra deaths come from where it's treated.

In [25]:
race_or = checks.race_odds_ratios(df, preds["lr_full"])
race_or.to_csv(RESULTS / "race_odds_ratios.csv")
race_or

ignoring hospital                within hospital  \
                              odds_ratio ci_low ci_high      odds_ratio   
Black/African American             1.217  1.155   1.282           1.022   
Multi-racial                       0.876  0.735   1.045           0.980   
Other Race                         1.052  1.005   1.101           0.951   

                                       
                       ci_low ci_high  
Black/African American  0.961   1.087  
Multi-racial            0.817   1.175  
Other Race              0.900   1.006